In [77]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
import sys
from pathlib import Path
from datasets import Dataset, DatasetDict
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

NOTEBOOK_ROOT = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_ROOT.parent if NOTEBOOK_ROOT.name == "src" else NOTEBOOK_ROOT
SRC_ROOT = PROJECT_ROOT / "src"

for import_root in (PROJECT_ROOT, SRC_ROOT):
    if str(import_root) not in sys.path:
        sys.path.append(str(import_root))

from dataset_loader import load_prolog_samples
from data.pipeline_constants import MAX_TRAIN_SAMPLE_TOKENS
import os

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
model_dtype = torch.bfloat16 if use_bf16 else torch.float16
DATASET_MAX_TRAIN_SAMPLE_TOKENS = int(
    os.getenv("DATASET_MAX_TRAIN_SAMPLE_TOKENS", str(MAX_TRAIN_SAMPLE_TOKENS))
)

In [78]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [79]:
model_id = os.getenv('TRAIN_BASE_MODEL_ID')
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=model_dtype,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype=model_dtype,
    device_map="auto",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
old_base_test_messages = [
    {"role": "system", "content": "Ты пишешь только код на Prolog."},
    {"role": "user", "content": "Реализуй предикат member_of(Element, List), который истинен, если элемент входит в список."},
]
base_test_messages = [
    {"role": "system", "content": "Ты пишешь только код на SWI-Prolog без пояснений."},
    {
        "role": "user",
        "content": "Реализуй предикат last_element(List, Element), который истинен, если Element — последний элемент списка. Верни только код.",
    },
]
model.device

Loading weights: 100%|██████████| 320/320 [00:02<00:00, 123.11it/s]


device(type='cuda', index=0)

In [80]:
inputs = tokenizer.apply_chat_template(
    base_test_messages,
    tokenize=True,
    enable_thinking=False,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

# Генерируем ответ
generated_ids = model.generate(**inputs, max_new_tokens=160)
new_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]
print(tokenizer.batch_decode(new_tokens, skip_special_tokens=True))

['last_element([], _).\nlast_element([Head|Tail], Last) :- Last is Head, Last is Tail.\n']


In [81]:
print(model.device)

cuda:0


In [82]:
tokenizer.eos_token_id

248046

In [83]:
tokenizer.pad_token_id

248044

In [84]:
def load_prolog_data_legacy(data_dir):
    """
    Загружает .pl файлы и соответствующие .txt описания в формате JSON.
    Структура: data_dir/annotated_repos/repo_name/*.pl и *.txt
    """
    data = []
    repos_path = Path(data_dir) / "annotated_repos"
    
    if not repos_path.exists():
        raise ValueError(f"Папка {repos_path} не найдена")
    
    for repo_dir in repos_path.iterdir():
        if not repo_dir.is_dir():
            continue
            
        print(f"📁 Обрабатываю репозиторий: {repo_dir.name}")
        
        # Проходим по всем .pl файлам в репозитории
        for pl_file in repo_dir.glob('*.pl'):
            txt_file = pl_file.with_suffix('.txt')
            
            # Проверяем наличие .txt файла
            if not txt_file.exists():
                print(f"  ⚠️  Нет описания для {pl_file.name} — пропускаю")
                continue
            
            description = txt_file.read_text(encoding='utf-8').strip()
                
            try:
                code = pl_file.read_text(encoding='utf-8').strip()
            except Exception as e:
                print(f"  ❌ Ошибка чтения {pl_file.name}: {e}")
                continue
            
            messages = [
                        {
                            "role": "system",
                            "content": (
                            "Вы — экспертный разработчик на Prolog. "
                            "Генерируйте чистый, идиоматичный код на Prolog на основе описания. "
        
                        )
                        },
                        {
                            "role": "user",
                            "content": description
                        },
                        {
                            "role": "assistant",
                            "content": f"\n{code}\n"
                        }
                    ]
            
            data.append(messages)
            
            
    
    print(f"\n📊 Всего загружено примеров: {len(data)}")
    return data


def load_dataset(data_dir, max_train_sample_tokens=DATASET_MAX_TRAIN_SAMPLE_TOKENS):
    samples = load_prolog_samples(
        data_dir,
        max_train_sample_tokens=max_train_sample_tokens,
        recalc_token_counts=True,
    )
    print(
        f"Loaded samples: {len(samples)} (token limit: {max_train_sample_tokens})"
    )
    return samples

In [85]:
# === Использование ===

train_samples = load_dataset('../data')




Loaded samples: 3031 (token limit: 2048)


In [86]:
print(len(train_samples))

3031


In [87]:
def render_full_text(messages):
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        enable_thinking=False,
        add_generation_prompt=False,
    )

def tokenize_messages(messages):
    full_text = render_full_text(messages)
    full_ids = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_TRAIN_SAMPLE_TOKENS,
    )["input_ids"]

    prefix_text = tokenizer.apply_chat_template(
        messages[:-1],
        tokenize=False,
        enable_thinking=False,
        add_generation_prompt=True,
    )
    prefix_ids = tokenizer(
        prefix_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_TRAIN_SAMPLE_TOKENS,
    )["input_ids"]

    prefix_len = min(len(prefix_ids), len(full_ids))
    labels = [-100] * prefix_len + full_ids[prefix_len:]

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

train_texts = [render_full_text(sample.messages) for sample in train_samples]
dataset_texts = train_texts

dataset = DatasetDict({
    "train": Dataset.from_list([{"messages": sample.messages} for sample in train_samples]),
})

In [88]:
import numpy as np
lengths = [len(tokenizer.encode(txt)) for txt in dataset_texts]
np.percentile(lengths, 95)

np.float64(1440.0)

In [89]:
def tokenize_func(examples):
    rows = [tokenize_messages(messages) for messages in examples["messages"]]
    return {
        "input_ids": [row["input_ids"] for row in rows],
        "attention_mask": [row["attention_mask"] for row in rows],
        "labels": [row["labels"] for row in rows],
    }

dataset = dataset.map(tokenize_func, batched=True, remove_columns=["messages"])
print(f"Train: {len(dataset['train'])}")

Map: 100%|██████████| 3031/3031 [00:03<00:00, 935.55 examples/s] 

Train: 3031


In [90]:
print(tokenizer.decode(dataset['train'][50]["input_ids"]))

<|im_start|>system
Вы - экспертный разработчик на Prolog. Генерируйте чистый, идиоматичный код на Prolog на основе описания.<|im_end|>
<|im_start|>user
Файл реализует обработчики HTTP-запросов для веб-приложения, используя библиотеки http_dispatch и html_write, где предикаты http_handler/3 регистрируют маршруты и связывают их с функциями отображения страниц, а предикаты styled/0 и mailman_demo/1 генерируют HTML-страницы с различными структурами и содержимым, включая навигационные элементы, созданные динамически на основе данных, заданных предикатами nav/2 и преобразованных через as_top_nav/2 и as_bottom_nav/2.<|im_end|>
<|im_start|>assistant
<think>

</think>

:- module(styling_handlers, []).

:- use_module(library(http/http_dispatch)).
:- use_module(library(http/html_write)).

user:body(strangeloop_style, Body) -->
        html(body([ div(id(top), h1('The Simple Web Page Site')),
                    div(id(content), Body)
                  ])).

:- http_handler(root(styled), styled , 

In [91]:
OUTPUT_DIR = "./prolog_model"           # Куда сохранить результат
EPOCHS = 3                           # Сколько раз прогнать данные
BATCH_SIZE = 1   

In [92]:
for name, _ in model.named_modules():
    print(name)


model
model.embed_tokens
model.layers
model.layers.0
model.layers.0.linear_attn
model.layers.0.linear_attn.act
model.layers.0.linear_attn.conv1d
model.layers.0.linear_attn.norm
model.layers.0.linear_attn.out_proj
model.layers.0.linear_attn.in_proj_qkv
model.layers.0.linear_attn.in_proj_z
model.layers.0.linear_attn.in_proj_b
model.layers.0.linear_attn.in_proj_a
model.layers.0.mlp
model.layers.0.mlp.gate_proj
model.layers.0.mlp.up_proj
model.layers.0.mlp.down_proj
model.layers.0.mlp.act_fn
model.layers.0.input_layernorm
model.layers.0.post_attention_layernorm
model.layers.1
model.layers.1.linear_attn
model.layers.1.linear_attn.act
model.layers.1.linear_attn.conv1d
model.layers.1.linear_attn.norm
model.layers.1.linear_attn.out_proj
model.layers.1.linear_attn.in_proj_qkv
model.layers.1.linear_attn.in_proj_z
model.layers.1.linear_attn.in_proj_b
model.layers.1.linear_attn.in_proj_a
model.layers.1.mlp
model.layers.1.mlp.gate_proj
model.layers.1.mlp.up_proj
model.layers.1.mlp.down_proj
model.

In [93]:
target_modules = ["q_proj", "k_proj", "v_proj"]


In [94]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules,
    task_type="CAUSAL_LM",
    
)
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=1e-5,
    gradient_accumulation_steps=4,  # Эффективный batch size будет 16
    lr_scheduler_type="linear",
    
    warmup_ratio=0.05,
    logging_steps=4,
    save_strategy="epoch",
    bf16=use_bf16,
    fp16=not use_bf16,
    report_to="none"          # Отключить лишние логи
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [95]:
def sft_collator(features):
    batch = tokenizer.pad(
        [
            {
                "input_ids": feature["input_ids"],
                "attention_mask": feature["attention_mask"],
            }
            for feature in features
        ],
        padding=True,
        return_tensors="pt",
    )
    max_length = batch["input_ids"].shape[1]
    labels = [
        feature["labels"] + [-100] * (max_length - len(feature["labels"]))
        for feature in features
    ]
    batch["labels"] = torch.tensor(labels, dtype=torch.long)
    return batch

In [96]:
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model,lora_config)
model.config.use_cache = False

In [97]:
model.print_trainable_parameters()

trainable params: 540,672 || all params: 1,882,365,760 || trainable%: 0.0287


In [98]:
print(dataset["train"][0].keys())

dict_keys(['input_ids', 'attention_mask', 'labels'])


In [99]:
model.gradient_checkpointing_enable()  

In [100]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

In [101]:
trainer = Trainer(model,args=training_args,data_collator=sft_collator,train_dataset=dataset['train'])

In [102]:
trainer._move_model_to_device(model = model,device=device)

In [103]:
print(os.getenv('TRAIN_BASE_MODEL_ID'))

Qwen/Qwen3.5-2B


In [104]:
trainer.train()

Step,Training Loss
4,1.387197
8,1.300493
12,1.227339
16,1.173650
20,1.372588
24,1.255432
28,0.970512
32,1.272183
36,1.476322
40,1.153891


TrainOutput(global_step=2274, training_loss=1.0969136995292066, metrics={'train_runtime': 10549.1799, 'train_samples_per_second': 0.862, 'train_steps_per_second': 0.216, 'total_flos': 4.21582113105984e+16, 'train_loss': 1.0969136995292066, 'epoch': 3.0})

In [105]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import os

base_model_id = os.getenv('TRAIN_BASE_MODEL_ID')
adapter_path = "./prolog_model/checkpoint-2274"  # или "./prolog_model", если там уже лежит финальная модель

print("Загружаю базовую модель и токенизатор...")
tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id, torch_dtype="auto", device_map="cpu", trust_remote_code=True
)

print("Загружаю и объединяю LoRA-адаптер...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model = model.merge_and_unload()

merged_path = "./prolog_merged"
print(f"Сохраняю полную модель в {merged_path}...")
model.save_pretrained(merged_path)
tokenizer.save_pretrained(merged_path)
print("Объединение завершено!")

Загружаю базовую модель и токенизатор...


Loading weights: 100%|██████████| 320/320 [00:00<00:00, 4922.19it/s]


Загружаю и объединяю LoRA-адаптер...
Сохраняю полную модель в ./prolog_merged...


Writing model shards: 100%|██████████| 1/1 [02:11<00:00, 131.01s/it]


Объединение завершено!
